# Avito Bootcamp — ориентация текстовых кропов

Нужно оценить `p_180` — вероятность того, что текст на изображении повернут на 180°. Используется официальный предобученный классификатор **PP-LCNet x1.0 textline orientation** из PaddlePaddle, ревизия весов зафиксирована как `cd237a4`. Дообучение не выполнялось.

Проверялись single-pass, согласование с поворотом на 180°, геометрический deskew и angle-TTA. Усреднение с поворотом на 180° отклонено из-за нарушения эквивариантности модели, а deskew — из-за ошибок сегментации и повторной интерполяции. Финальный метод выполняет три локальных прохода для углов `−15°, 0°, +15°` и усредняет логиты. Это делает ответ устойчивее к небольшому наклону без обучения на тестовых данных. Калибровка не применялась, поскольку размеченной реальной validation-выборки нет.

Техническая валидация содержит 432 синтетических примера: 8 русских и английских строк, 3 шрифта, классы 0°/180° и наклоны от −20° до +20°. Single-pass получил accuracy **0.8935** и Brier **0.078354**; выбранный 3-angle TTA — accuracy **0.9051** и Brier **0.070480** (`1 − Brier = 0.929520`). Эта проверка подтверждает улучшение устойчивости к наклону; файл весов занимает около **6.4 MiB**.

Источники open-source компонентов: [модель PP-LCNet x1.0](https://huggingface.co/PaddlePaddle/PP-LCNet_x1_0_textline_ori) и [документация PaddleOCR](https://www.paddleocr.ai/main/en/version3.x/module_usage/textline_orientation_classification.html). Модель скачивается один раз, весь inference выполняется локально; внешние inference API не используются.

## Dependencies

Ячейка ниже автоматически определяет операционную систему и наличие NVIDIA GPU. На Windows/Linux с доступной NVIDIA GPU устанавливается PaddlePaddle GPU для CUDA 12.6; иначе используется CPU-сборка. На Windows с GPU дополнительно устанавливается совместимый cuDNN. Остальные зависимости устанавливаются одинаково для всех систем, поэтому ноутбук можно запускать через Run All без ручного раскомментирования команд.

In [1]:
import importlib.metadata
import platform
import shutil
import subprocess
import sys
import struct
from pathlib import Path

PADDLE_VERSION = "3.3.0"
PYPI_INDEX = "https://pypi.org/simple"

COMMON_PACKAGES = [
    "numpy==1.26.4",
    "opencv-python-headless==4.10.0.84",
    "PyYAML==6.0.2",
    "Pillow==11.3.0",
]

system = platform.system()
machine = platform.machine()
python_bits = struct.calcsize("P") * 8
python_version = platform.python_version()

print(f"OS: {system}")
print(f"Architecture: {machine}")
print(f"Python: {python_version} ({python_bits}-bit)")
print(f"Executable: {sys.executable}")

if system == "Windows" and python_bits != 64:
    raise RuntimeError(
        "\nPaddlePaddle requires 64-bit Python on Windows.\n"
        "Install 64-bit Python and select its Jupyter kernel.\n"
        f"Current interpreter: {sys.executable}"
    )

def run_command(command):
    print("\n>", " ".join(command))

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(result.stdout)

    if result.returncode != 0:
        lines = [line for line in result.stdout.splitlines() if line.strip()]
        tail = "\n".join(lines[-30:])

        raise RuntimeError(
            f"\nCommand failed with exit code {result.returncode}:\n"
            + " ".join(command)
            + "\n\nLast pip output:\n"
            + tail
        )

def pip_install(*args):
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            *args,
        ]
    )

def installed_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None

def ensure_package(package_spec, package_name=None, extra_args=None):
    package_name = package_name or package_spec.split("==")[0]
    required_version = package_spec.split("==")[1] if "==" in package_spec else None
    current_version = installed_version(package_name)

    if current_version and (
        required_version is None or current_version == required_version
    ):
        print(f"{package_name} {current_version} already installed -> skipping")
        return

    if package_name in {"paddlepaddle", "paddlepaddle-gpu"} and "paddle" in sys.modules:
        raise RuntimeError(
            "\nPaddlePaddle is already loaded in this Jupyter kernel, "
            "so Windows cannot replace libpaddle.pyd.\n"
            "Restart the kernel once and run all cells from the beginning."
        )

    args = [package_spec]
    if extra_args:
        args.extend(extra_args)

    pip_install(*args)

# Update pip only if needed by the environment.
pip_install("pip", "--index-url", PYPI_INDEX)

has_nvidia_gpu = False

if system in {"Windows", "Linux"}:
    nvidia_smi = shutil.which("nvidia-smi")

    if nvidia_smi is None and system == "Windows":
        possible_path = r"C:\Windows\System32\nvidia-smi.exe"
        if Path(possible_path).exists():
            nvidia_smi = possible_path

    if nvidia_smi:
        try:
            result = subprocess.run(
                [nvidia_smi, "-L"],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=10,
                check=False,
            )
            has_nvidia_gpu = (
                result.returncode == 0
                and bool(result.stdout.strip())
            )
        except (OSError, subprocess.SubprocessError):
            has_nvidia_gpu = False

print(f"NVIDIA GPU: {'found' if has_nvidia_gpu else 'not found'}")

# Common dependencies.
for spec in COMMON_PACKAGES:
    name = spec.split("==")[0]
    ensure_package(
        spec,
        package_name=name,
        extra_args=["--index-url", PYPI_INDEX],
    )

if has_nvidia_gpu:
    print("\nChecking PaddlePaddle GPU...")

    ensure_package(
        f"paddlepaddle-gpu=={PADDLE_VERSION}",
        package_name="paddlepaddle-gpu",
        extra_args=[
            "-i",
            "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        ],
    )

    if system == "Windows":
        print("\nChecking cuDNN...")
        ensure_package(
            "nvidia-cudnn-cu12==9.9.0.52",
            package_name="nvidia-cudnn-cu12",
            extra_args=[
                "--no-deps",
                "--index-url",
                PYPI_INDEX,
            ],
        )

else:
    print("\nChecking PaddlePaddle CPU...")

    ensure_package(
        f"paddlepaddle=={PADDLE_VERSION}",
        package_name="paddlepaddle",
        extra_args=[
            "--index-url",
            PYPI_INDEX,
            "--only-binary=:all:",
        ],
    )

print("\nDependencies are ready.")


OS: Windows
Architecture: AMD64
Python: 3.9.13 (64-bit)
Executable: c:\Users\mrlog\AppData\Local\Programs\Python\Python39\python.exe

> c:\Users\mrlog\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip --index-url https://pypi.org/simple

NVIDIA GPU: not found
numpy 1.26.4 already installed -> skipping
opencv-python-headless 4.10.0.84 already installed -> skipping
PyYAML 6.0.2 already installed -> skipping
Pillow 11.3.0 already installed -> skipping

Checking PaddlePaddle CPU...

> c:\Users\mrlog\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade paddlepaddle==3.3.0 --index-url https://pypi.org/simple --only-binary=:all:
  Using cached paddlepaddle-3.3.0-cp39-cp39-win_amd64.whl.metadata (8.8 kB)
Using cached paddlepaddle-3.3.0-cp39-cp39-win_amd64.whl (104.2 MB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Dependencies are ready.


## Imports

Импортируем библиотеки, фиксируем пути и параметры запуска. Все остальные ячейки используют только определения из этого ноутбука.

In [2]:
import csv
import hashlib
import os
import random
import platform
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

SEED = 2026
os.environ.setdefault('PYTHONHASHSEED', str(SEED))
os.environ.setdefault('FLAGS_cudnn_deterministic', '1')

import cv2
import numpy as np
import paddle
import yaml
from PIL import Image, ImageDraw, ImageFont
from paddle.inference import Config, create_predictor

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / 'dataset'
IMAGES_DIR = DATA_DIR / 'test' / 'images'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'
MODEL_DIR = ROOT / '.cache' / 'model'
SUBMISSION_PATH = ROOT / 'submission.csv'

MODEL_REPO = 'PaddlePaddle/PP-LCNet_x1_0_textline_ori'
MODEL_REVISION = 'cd237a4'
MODEL_FILES = ('inference.json', 'inference.pdiparams', 'inference.yml', 'config.json')
BATCH_SIZE = 64
TTA_ANGLES = (-15, 0, 15)
LOGIT_EPS = 1e-6
LOADER_WORKERS = 4
CPU_THREADS = 4
GPU_MEMORY_MB = 1024
DEVICE_ID = 0

random.seed(SEED)
np.random.seed(SEED)
paddle.seed(SEED)
cv2.setRNGSeed(SEED)

print('Python:', platform.python_version())
print('Paddle:', paddle.__version__)
print('CUDA build:', paddle.device.is_compiled_with_cuda())

c:\Users\mrlog\AppData\Local\Programs\Python\Python39\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Python: 3.9.13
Paddle: 3.3.0
CUDA build: False


## Input contract

Проверяем только структуру входных данных: колонки CSV, уникальность ID и наличие ровно одного PNG для каждого ID. Содержание тестовых изображений не исследуется.

In [3]:
with SAMPLE_PATH.open('r', encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == ['image_id', 'p_180']
    image_ids = [row['image_id'].strip() for row in reader]

assert image_ids and len(image_ids) == len(set(image_ids))
image_paths = [IMAGES_DIR / f'{image_id}.png' for image_id in image_ids]
missing = [path.name for path in image_paths if not path.is_file()]
actual_ids = {path.stem for path in IMAGES_DIR.glob('*.png')}
assert not missing, f'Missing images: {missing[:10]}'
assert actual_ids == set(image_ids)
print(f'Input contract OK: {len(image_ids)} ordered images')

Input contract OK: 20000 ordered images


## Model files

Скачиваем четыре официальных файла модели из зафиксированной ревизии. Для воспроизводимости вычисляем SHA-256 каждого файла. Повторный запуск использует локальный кэш.

In [4]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_hashes = {}
for filename in MODEL_FILES:
    destination = MODEL_DIR / filename
    if not destination.is_file():
        url = f'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{filename}'
        request = urllib.request.Request(url, headers={'User-Agent': 'avito-orientation-notebook'})
        with urllib.request.urlopen(request, timeout=180) as response:
            destination.write_bytes(response.read())
    model_hashes[filename] = hashlib.sha256(destination.read_bytes()).hexdigest()

expected_hashes = {
    'inference.json': 'b81929eeaff8e52db0fafb49c9fbfc5bc0572f81641fe0179cc52096323fb4d4',
    'inference.pdiparams': '0de2bcf996cf553e2b848dd7b1769dafffc6917b1ccdf55c1d8efe7909fbf743',
    'inference.yml': '8d5120d0e1a30a9df7ed46aa9119da3796ed066777089d1c1d705f132d5e90f9',
    'config.json': 'c1a2576f9713adabdc4a58d9d654fcc9ea93b6efd0e2d3581d5aab409a33f14f',
}
assert model_hashes == expected_hashes, 'Model checksum mismatch'
model_hashes

{'inference.json': 'b81929eeaff8e52db0fafb49c9fbfc5bc0572f81641fe0179cc52096323fb4d4',
 'inference.pdiparams': '0de2bcf996cf553e2b848dd7b1769dafffc6917b1ccdf55c1d8efe7909fbf743',
 'inference.yml': '8d5120d0e1a30a9df7ed46aa9119da3796ed066777089d1c1d705f132d5e90f9',
 'config.json': 'c1a2576f9713adabdc4a58d9d654fcc9ea93b6efd0e2d3581d5aab409a33f14f'}

## Preprocessing

Берём размер входа, нормализацию и порядок классов непосредственно из `inference.yml`. Дополнительно проверяем, что второй класс действительно означает `180_degree`.

In [5]:
model_config = yaml.safe_load((MODEL_DIR / 'inference.yml').read_text(encoding='utf-8'))
operations = model_config['PreProcess']['transform_ops']
labels = model_config['PostProcess']['Topk']['label_list']
assert labels == ['0_degree', '180_degree']

resize = next(item['ResizeImage'] for item in operations if 'ResizeImage' in item)
normalization = next(item['NormalizeImage'] for item in operations if 'NormalizeImage' in item)
SIZE = tuple(resize['size'])
MEAN = np.asarray(normalization['mean'], dtype=np.float32)
STD = np.asarray(normalization['std'], dtype=np.float32)
SCALE = np.float32(normalization['scale'])
CLASS_180 = labels.index('180_degree')

assert SIZE == (160, 80)
assert MEAN.shape == STD.shape == (3,)
print('Input width, height:', SIZE, '| labels:', labels)

Input width, height: (160, 80) | labels: ['0_degree', '180_degree']


## Predictor

Инициализируем официальный Paddle predictor. GPU выбирается автоматически, если установлен CUDA-вариант Paddle; иначе используется CPU.

In [6]:
inference_config = Config(
    str(MODEL_DIR / 'inference.json'),
    str(MODEL_DIR / 'inference.pdiparams'),
)
if paddle.device.is_compiled_with_cuda():
    inference_config.enable_use_gpu(GPU_MEMORY_MB, DEVICE_ID)
    device = 'gpu'
else:
    inference_config.disable_gpu()
    inference_config.set_cpu_math_library_num_threads(CPU_THREADS)
    device = 'cpu'
inference_config.disable_glog_info()

predictor = create_predictor(inference_config)
input_handle = predictor.get_input_handle(predictor.get_input_names()[0])
output_handle = predictor.get_output_handle(predictor.get_output_names()[0])
print('Inference device:', device)

Inference device: cpu


## Inference functions

Декодируем PNG в RGB, применяем предобработку из YAML и получаем вероятность класса `180_degree`. Для устойчивости к наклону создаём три детерминированных представления `−15°, 0°, +15°`; фон новых углов заполняется медианным цветом границы изображения. Итог — sigmoid среднего логита.

In [7]:
def read_rgb(path):
    encoded = np.fromfile(path, dtype=np.uint8)
    bgr = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Cannot decode {path}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def preprocess(rgb):
    resized = cv2.resize(rgb, SIZE, interpolation=cv2.INTER_LINEAR)
    normalized = (resized.astype(np.float32) * SCALE - MEAN) / STD
    return np.ascontiguousarray(normalized.transpose(2, 0, 1))

def predict_tensor(batch):
    batch = np.ascontiguousarray(batch, dtype=np.float32)
    input_handle.reshape(batch.shape)
    input_handle.copy_from_cpu(batch)
    predictor.run()
    probabilities = np.asarray(output_handle.copy_to_cpu())
    assert probabilities.shape == (len(batch), 2)
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)
    return probabilities[:, CLASS_180].astype(np.float64)

def predict_images(rgb_images):
    tensors = np.stack([preprocess(image) for image in rgb_images])
    probabilities = predict_tensor(tensors)
    assert np.isfinite(probabilities).all()
    assert ((0 <= probabilities) & (probabilities <= 1)).all()
    return probabilities

def rotate_for_tta(rgb, angle):
    if angle == 0:
        return rgb
    border = np.concatenate([rgb[0], rgb[-1], rgb[:, 0], rgb[:, -1]], axis=0)
    fill = tuple(np.median(border, axis=0).astype(np.uint8))
    image = Image.fromarray(rgb)
    return np.asarray(image.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True, fillcolor=fill))

def predict_angle_tta(rgb_images):
    predictions = []
    for angle in TTA_ANGLES:
        augmented = [rotate_for_tta(image, angle) for image in rgb_images]
        predictions.append(predict_images(augmented))
    probabilities = np.stack(predictions, axis=1)
    clipped = np.clip(probabilities, LOGIT_EPS, 1.0 - LOGIT_EPS)
    logits = np.log(clipped / (1.0 - clipped))
    mean_logits = logits.mean(axis=1)
    return 1.0 / (1.0 + np.exp(-mean_logits))

## Synthetic validation

Создаём строки с известной ориентацией, тремя шрифтами и наклонами от −20° до +20°. Сравниваем single-pass с выбранным angle-TTA по accuracy и основной метрике задачи — Brier Score

In [8]:
font_candidates = [
    [Path(r'C:/Windows/Fonts/arial.ttf'), Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')],
    [Path(r'C:/Windows/Fonts/times.ttf'), Path('/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf')],
    [Path(r'C:/Windows/Fonts/calibri.ttf'), Path('/usr/share/fonts/truetype/dejavu/DejaVuSansCondensed.ttf')],
]
font_paths = [next(path for path in alternatives if path.is_file()) for alternatives in font_candidates]
phrases = [
    'Hello world', 'Text orientation', 'Avito Bootcamp', 'Sample number 42',
    'Продается велосипед', 'Доставка сегодня', 'Новая квартира', 'Телефон в наличии',
]
validation_skews = (-20, -15, -10, -5, 0, 5, 10, 15, 20)
validation_images, validation_labels = [], []
for font_path in font_paths:
    font = ImageFont.truetype(str(font_path), 32)
    for phrase in phrases:
        box = font.getbbox(phrase)
        image = Image.new('RGB', (box[2] - box[0] + 32, box[3] - box[1] + 32), 'white')
        ImageDraw.Draw(image).text((16 - box[0], 16 - box[1]), phrase, font=font, fill='black')
        for label in (0, 1):
            oriented = image if label == 0 else image.rotate(180, resample=Image.Resampling.BICUBIC, expand=True, fillcolor='white')
            for skew in validation_skews:
                skewed = oriented.rotate(skew, resample=Image.Resampling.BICUBIC, expand=True, fillcolor='white')
                validation_images.append(np.asarray(skewed))
                validation_labels.append(label)

single_pass_p = predict_images(validation_images)
validation_p = predict_angle_tta(validation_images)
repeat_validation_p = predict_angle_tta(validation_images)
np.testing.assert_allclose(validation_p, repeat_validation_p, rtol=0, atol=1e-7)
validation_y = np.asarray(validation_labels)
single_accuracy = float(np.mean((single_pass_p >= 0.5) == validation_y))
single_brier = float(np.mean((single_pass_p - validation_y) ** 2))
accuracy = float(np.mean((validation_p >= 0.5) == validation_y))
brier = float(np.mean((validation_p - validation_y) ** 2))
ranks = np.empty(len(validation_p), dtype=float)
ranks[np.argsort(validation_p)] = np.arange(1, len(validation_p) + 1)
n_positive = int(validation_y.sum())
n_negative = len(validation_y) - n_positive
roc_auc = float((ranks[validation_y == 1].sum() - n_positive * (n_positive + 1) / 2) / (n_positive * n_negative))

validation_metrics = {
    'samples': len(validation_y),
    'single_accuracy': single_accuracy,
    'single_brier': single_brier,
    'tta_accuracy': accuracy,
    'tta_brier': brier,
    'tta_1_minus_brier': 1.0 - brier,
    'tta_roc_auc': roc_auc,
}
validation_metrics

{'samples': 432,
 'single_accuracy': 0.8935185185185185,
 'single_brier': 0.07835445893111632,
 'tta_accuracy': 0.9050925925925926,
 'tta_brier': 0.07047996361050253,
 'tta_1_minus_brier': 0.9295200363894974,
 'tta_roc_auc': 0.9739797668038409}

### Интерпретация метрик

На 432 контролируемых примерах angle-TTA улучшает и accuracy, и Brier относительно single-pass. Повторный запуск validation дополнительно проверяет воспроизводимость. Синтетика всё ещё проще реальных кропов, поэтому мы не переносим полученные числа на скрытый тест. Честный test Brier Score станет известен только после оценки submission платформой.

## Full inference

Для каждого ID в порядке `sample_submission.csv` выполняем три прохода `−15°, 0°, +15°` и объединяем логиты. Обработка батчами ограничивает расход памяти; чтение изображений распараллелено.

In [9]:
all_probabilities = []
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=LOADER_WORKERS) as pool:
    for start in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        rgb_batch = list(pool.map(read_rgb, batch_paths))
        all_probabilities.append(predict_angle_tta(rgb_batch))

p_180 = np.concatenate(all_probabilities)
elapsed = time.perf_counter() - started
print(f'Processed {len(p_180)} images on {device} in {elapsed:.1f}s ({len(p_180)/elapsed:.1f} images/s)')

Processed 20000 images on cpu in 439.5s (45.5 images/s)


## Submission

Сохраняем submission с двумя требуемыми колонками. После записи проверяем количество строк, порядок ID, отсутствие пропусков и диапазон вероятностей.

In [10]:
with SUBMISSION_PATH.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle, lineterminator='\n')
    writer.writerow(['image_id', 'p_180'])
    writer.writerows((image_id, f'{probability:.10f}') for image_id, probability in zip(image_ids, p_180))

with SUBMISSION_PATH.open('r', encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
saved_ids = [row['image_id'] for row in rows]
saved_probabilities = np.asarray([float(row['p_180']) for row in rows])
assert len(rows) == len(image_ids) == 20_000
assert saved_ids == image_ids
assert np.isfinite(saved_probabilities).all()
assert ((0 <= saved_probabilities) & (saved_probabilities <= 1)).all()
np.testing.assert_allclose(saved_probabilities, p_180, rtol=0, atol=5.1e-11)
repeat_count = min(BATCH_SIZE, len(image_paths))
repeat_images = [read_rgb(path) for path in image_paths[:repeat_count]]
repeat_probabilities = predict_angle_tta(repeat_images)
np.testing.assert_allclose(repeat_probabilities, p_180[:repeat_count], rtol=0, atol=1e-7)

print('Saved:', SUBMISSION_PATH)
print('Rows:', len(rows))
print('Probability range:', float(saved_probabilities.min()), float(saved_probabilities.max()))

Saved: D:\dop_study\avito\flip_check\submission.csv
Rows: 20000
Probability range: 1.4213e-06 0.999999
